### https://www.kaggle.com/datasets/msambare/fer2013

# Classification d’expressions faciales avec Régressions logistiques

## Dataset FER2013

* Membre du groupe :
Thomas Lefebvre ; 
Junior ; 
Jack ; 
Kemuel ; 
Mory ; 
Adolphe


## Présentation du projet

Ce projet a pour objectif de déveleopper un modèle de classification d'expressions faciales sur le dataset FER2013.
On commencera par télécharger le dataset depuis kaggle, et on le mettra dans un dataloader au vu de la taille du dataset.


Le modèle utilise est une régression logistique multinomiale. Le dataset utilisé est le FER2013.Pytorch sera utilisé pour le chargement et pour l'entrainement.

Je n'ai pas utilisé de PCA dans la premier test 


** Objectif final : **

Construire un modèle mutli classe performant basé sur un algorithme simple, interprétable et efficace.


In [1]:
import sys
from pathlib import Path

# Le noyau Jupyter peut démarrer à la racine du dépôt ou dans notebooks/.
racine = Path.cwd()
while not (racine / "src" / "donnees.py").is_file():
    if racine == racine.parent:
        raise FileNotFoundError("Ouvrez le notebook depuis le dépôt ML.")
    racine = racine.parent
if str(racine) not in sys.path:
    sys.path.insert(0, str(racine))

from src.donnees import creer_dataloaders

## Chargement du data set

In [2]:
train_loader, val_loader, test_loader = creer_dataloaders(batch_size=64)

print(f"Train : {len(train_loader.dataset)} images")
print(f"Validation : {len(val_loader.dataset)} images")
print(f"Test : {len(test_loader.dataset)} images")

Train : 24402 images
Validation : 4307 images
Test : 7178 images


Nous avons dans le dataset , un dossier train et un dossier test.
Le data set validation permettra une première évaluation de notre modèle avec l'ajustement des hyperparamètres.
Le data set test permettra une évaluation finale de notre modèle.


## Vérification des lots

In [3]:
print("Train :", len(train_loader))
print("Validation :", len(val_loader))
print("Test :", len(test_loader))

Train : 382
Validation : 68
Test : 113


Ces vérifications sont cohérentes nous avons créer des lots de 64 images.


On a mis shuffle = True pour le train afin que à chaque epoch les images soient dans un ordre différent.


On a mis shuffle = False pour le validation et le test afin de pouvoir comparer les résultats de manière plus stable.


## Regressions Logistiques

Avec pytorch on utilise un modèle linéaire que l'on adapte directement.

In [4]:
import torch
import torch.nn as nn

class LogisticRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(48 * 48, 7)

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        return self.linear(x)

In [5]:
# Entrainement : 

model = LogisticRegression()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.LBFGS(model.parameters(), lr=0.1)

# Une version plus simple pour commencer :
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/20 - Loss: 1.7986
Epoch 2/20 - Loss: 1.7291
Epoch 3/20 - Loss: 1.7025
Epoch 4/20 - Loss: 1.6961
Epoch 5/20 - Loss: 1.6785
Epoch 6/20 - Loss: 1.6823
Epoch 7/20 - Loss: 1.6568
Epoch 8/20 - Loss: 1.6576
Epoch 9/20 - Loss: 1.6593
Epoch 10/20 - Loss: 1.6475
Epoch 11/20 - Loss: 1.6492
Epoch 12/20 - Loss: 1.6486
Epoch 13/20 - Loss: 1.6319
Epoch 14/20 - Loss: 1.6339
Epoch 15/20 - Loss: 1.6258
Epoch 16/20 - Loss: 1.6271
Epoch 17/20 - Loss: 1.6225
Epoch 18/20 - Loss: 1.6206
Epoch 19/20 - Loss: 1.6147
Epoch 20/20 - Loss: 1.6164


In [6]:
# Evaluation : 
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Validation accuracy : {accuracy:.2%}")

Validation accuracy : 34.39%


Premier résultat avec pytorch : sans aucune optimisation , accuracy = 32.9%


In [ ]:
from sklearn.metrics import confusion_matrix
y_pred = model.predict(X_test)
confusion_matrix(y_test, y_pred)


##revérifier



AttributeError: 'LogisticRegression' object has no attribute 'predict'